# UAV Privacy — Phase 0 & Phase 1
**Phase 0**: Khởi tạo, kiểm kê & chuẩn hoá 2 dataset → COCO → YOLO format

**Phase 1**: Train person detector (YOLOv8n single-class)

## Datasets
1. **License-Plate-10K** — `mtu2006/license-plate-10k`
2. **Birdseye-RU** — `mdahanafarifkhan/birdseye-ru`

> Make sure both datasets are added as **Input** to this notebook.

---
## 0. Install dependencies & setup

In [ ]:
!pip install -q ultralytics>=8.1.0 pycocotools>=2.0.7 pyarrow>=14.0.0 lightgbm>=4.1.0

In [ ]:
import os, sys, json, glob, shutil, random, time
import numpy as np
import pandas as pd
import cv2
import yaml
from pathlib import Path
from tqdm import tqdm

# ── Paths ──
WORK = '/kaggle/working'
INTERIM = f'{WORK}/data/interim'
PROCESSED = f'{WORK}/data/processed'
MANIFESTS = f'{WORK}/data/manifests'
ARTIFACTS = f'{WORK}/artifacts/detector'
REPORTS = f'{WORK}/reports'
VISUALS = f'{WORK}/artifacts/visuals'

for d in [INTERIM, PROCESSED, MANIFESTS, ARTIFACTS, REPORTS, VISUALS]:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')

---
## 1. Discover datasets

In [ ]:
# List available input datasets
INPUT = '/kaggle/input'
print('Available datasets:')
for entry in sorted(os.listdir(INPUT)):
    full = os.path.join(INPUT, entry)
    if os.path.isdir(full):
        n_files = sum(1 for _ in Path(full).rglob('*') if _.is_file())
        print(f'  📂 {entry}  ({n_files} files)')

In [ ]:
# Show first 2 levels of each dataset
DATASETS = {
    'license_plate_10k': '/kaggle/input/license-plate-10k',
    'birdseye_ru':       '/kaggle/input/birdseye-ru',
}

for name, root in DATASETS.items():
    print(f'\n=== {name} ===')
    if not os.path.isdir(root):
        print(f'  ⚠ NOT FOUND at {root}')
        # Try finding it
        for d in os.listdir(INPUT):
            if name.replace('_', '-') in d or name.replace('_', '') in d:
                alt = os.path.join(INPUT, d)
                print(f'  → Found alternative: {alt}')
                DATASETS[name] = alt
                root = alt
                break
    if os.path.isdir(root):
        for r, dirs, files in os.walk(root):
            level = r.replace(root, '').count(os.sep)
            indent = '  ' * (level + 1)
            print(f'{indent}{os.path.basename(r)}/')
            if level >= 1:
                sub_indent = '  ' * (level + 2)
                for f in files[:5]:
                    print(f'{sub_indent}{f}')
                if len(files) > 5:
                    print(f'{sub_indent}... and {len(files)-5} more')
                break

---
## 2. Phase 0 — Convert to COCO & create splits

In [ ]:
# ── Helper functions (inline for Kaggle) ──

def find_images(root):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp','*.tif','*.tiff')
    imgs = []
    for ext in exts:
        imgs.extend(glob.glob(os.path.join(root,'**',ext), recursive=True))
    return sorted(imgs)

def find_annotations(root):
    result = {'txt':[],'xml':[],'json':[],'csv':[]}
    for ext,key in [('*.txt','txt'),('*.xml','xml'),('*.json','json'),('*.csv','csv')]:
        result[key] = sorted(glob.glob(os.path.join(root,'**',ext), recursive=True))
    return result

def detect_format(ann):
    for jf in ann['json']:
        try:
            with open(jf) as f: data = json.load(f)
            if isinstance(data,dict) and 'images' in data: return 'coco'
        except: pass
    if len(ann['txt'])>10: return 'yolo'
    if len(ann['xml'])>10: return 'voc'
    if ann['csv']: return 'csv'
    return 'unknown'

print('Helper functions defined.')

In [ ]:
def convert_yolo_to_coco(img_paths, ann_dir, class_names=None):
    """Parse YOLO .txt → COCO dict."""
    cats = [{'id':i+1,'name':n} for i,n in enumerate(class_names or ['object'])]
    coco = {'info':{},'licenses':[],'images':[],'annotations':[],'categories':cats}
    aid = 1
    for iid, ip in enumerate(tqdm(img_paths, desc='YOLO→COCO'), 1):
        img = cv2.imread(ip)
        if img is None: continue
        h,w = img.shape[:2]
        coco['images'].append({'id':iid,'file_name':ip,'width':w,'height':h})
        stem = os.path.splitext(os.path.basename(ip))[0]
        txt = None
        for c in [os.path.join(ann_dir,stem+'.txt'), os.path.join(os.path.dirname(ip),stem+'.txt')]:
            if os.path.isfile(c): txt=c; break
        if txt is None: continue
        with open(txt) as f:
            for line in f:
                p = line.strip().split()
                if len(p)<5: continue
                ci=int(p[0]); cx,cy,bw,bh=map(float,p[1:5])
                aw,ah = bw*w, bh*h
                ax,ay = cx*w-aw/2, cy*h-ah/2
                coco['annotations'].append({'id':aid,'image_id':iid,'category_id':ci+1,
                    'bbox':[ax,ay,aw,ah],'area':aw*ah,'iscrowd':0})
                aid+=1
    return coco

def convert_voc_to_coco(img_paths, xml_files, class_names=None):
    """Parse VOC XML → COCO dict."""
    import xml.etree.ElementTree as ET
    cats = [{'id':1,'name':class_names[0] if class_names else 'object'}]
    coco = {'info':{},'licenses':[],'images':[],'annotations':[],'categories':cats}
    xmap = {os.path.splitext(os.path.basename(x))[0]:x for x in xml_files}
    aid = 1
    cat_map = {}
    for iid,ip in enumerate(tqdm(img_paths,desc='VOC→COCO'),1):
        stem = os.path.splitext(os.path.basename(ip))[0]
        img = cv2.imread(ip)
        if img is None: continue
        h,w = img.shape[:2]
        coco['images'].append({'id':iid,'file_name':ip,'width':w,'height':h})
        if stem not in xmap: continue
        for obj in ET.parse(xmap[stem]).getroot().findall('object'):
            nm = obj.find('name').text
            if nm not in cat_map:
                cid = len(cat_map)+1
                cat_map[nm]=cid
                if not any(c['name']==nm for c in coco['categories']):
                    coco['categories'].append({'id':cid,'name':nm})
            bb = obj.find('bndbox')
            x1,y1=float(bb.find('xmin').text),float(bb.find('ymin').text)
            x2,y2=float(bb.find('xmax').text),float(bb.find('ymax').text)
            coco['annotations'].append({'id':aid,'image_id':iid,'category_id':cat_map[nm],
                'bbox':[x1,y1,x2-x1,y2-y1],'area':(x2-x1)*(y2-y1),'iscrowd':0})
            aid+=1
    return coco

def convert_csv_to_coco(img_paths, csv_path, class_names=None):
    """Parse CSV → COCO dict."""
    cats = [{'id':1,'name':class_names[0] if class_names else 'object'}]
    coco = {'info':{},'licenses':[],'images':[],'annotations':[],'categories':cats}
    df = pd.read_csv(csv_path); df.columns=[c.strip().lower() for c in df.columns]
    im = {os.path.basename(p):p for p in img_paths}
    aid=1; cat_map={}
    for iid,(fn,g) in enumerate(tqdm(df.groupby('filename'),desc='CSV→COCO'),1):
        ip = im.get(fn)
        if ip is None: continue
        img = cv2.imread(ip)
        if img is None: continue
        h,w = img.shape[:2]
        coco['images'].append({'id':iid,'file_name':ip,'width':w,'height':h})
        for _,row in g.iterrows():
            x1=float(row.get('xmin',row.get('x_min',0))); y1=float(row.get('ymin',row.get('y_min',0)))
            x2=float(row.get('xmax',row.get('x_max',0))); y2=float(row.get('ymax',row.get('y_max',0)))
            cn = str(row.get('class',row.get('label','object')))
            if cn not in cat_map:
                cid=len(cat_map)+1; cat_map[cn]=cid
                if not any(c['name']==cn for c in coco['categories']):
                    coco['categories'].append({'id':cid,'name':cn})
            coco['annotations'].append({'id':aid,'image_id':iid,'category_id':cat_map[cn],
                'bbox':[x1,y1,x2-x1,y2-y1],'area':(x2-x1)*(y2-y1),'iscrowd':0})
            aid+=1
    return coco

print('Converter functions defined.')

In [ ]:
# ── Convert each dataset ──
all_coco = []
ds_names = []

for ds_name, ds_root in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'Converting: {ds_name}')
    print(f'Root: {ds_root}')
    
    if not os.path.isdir(ds_root):
        print(f'⚠ Skipping — directory not found.')
        continue
    
    images = find_images(ds_root)
    ann = find_annotations(ds_root)
    fmt = detect_format(ann)
    print(f'Found {len(images)} images, format: {fmt}')
    
    if fmt == 'coco':
        for jf in ann['json']:
            with open(jf) as f: data = json.load(f)
            if isinstance(data,dict) and 'images' in data:
                coco = data; break
    elif fmt == 'yolo':
        ldirs = {os.path.dirname(t) for t in ann['txt']}
        ad = sorted(ldirs)[0] if ldirs else ds_root
        coco = convert_yolo_to_coco(images, ad)
    elif fmt == 'voc':
        coco = convert_voc_to_coco(images, ann['xml'])
    elif fmt == 'csv':
        coco = convert_csv_to_coco(images, ann['csv'][0])
    else:
        print('Unknown format — indexing images only.')
        coco = {'info':{},'licenses':[],'images':[],'annotations':[],'categories':[{'id':1,'name':'object'}]}
        for iid,ip in enumerate(tqdm(images),1):
            img = cv2.imread(ip)
            if img is None: continue
            h,w = img.shape[:2]
            coco['images'].append({'id':iid,'file_name':ip,'width':w,'height':h})
    
    # Save individual COCO
    out = os.path.join(INTERIM, f'coco_{ds_name}.json')
    with open(out,'w') as f: json.dump(coco,f)
    print(f'✅ {len(coco["images"])} images, {len(coco["annotations"])} annotations → {out}')
    
    all_coco.append(coco)
    ds_names.append(ds_name)

print(f'\n✅ Converted {len(all_coco)} dataset(s).')

In [ ]:
# ── Merge datasets ──
def merge_coco(coco_list, names):
    merged = {'info':{'description':f'Merged: {", ".join(names)}'},'licenses':[],
              'images':[],'annotations':[],'categories':[]}
    cn2id = {}; cid = 1
    for c in coco_list:
        for cat in c['categories']:
            if cat['name'] not in cn2id:
                cn2id[cat['name']]=cid
                merged['categories'].append({'id':cid,'name':cat['name']})
                cid+=1
    ioff=0; aoff=0
    for c in coco_list:
        ocm = {cat['id']:cn2id[cat['name']] for cat in c['categories']}
        oim = {}
        for img in c['images']:
            nid = img['id']+ioff; oim[img['id']]=nid
            merged['images'].append({**img,'id':nid})
        for ann in c['annotations']:
            merged['annotations'].append({**ann,'id':ann['id']+aoff,
                'image_id':oim[ann['image_id']],'category_id':ocm.get(ann['category_id'],ann['category_id'])})
        ioff+=len(c['images'])+1; aoff+=len(c['annotations'])+1
    return merged

if len(all_coco)==1:
    merged = all_coco[0]
else:
    merged = merge_coco(all_coco, ds_names)

merged_path = os.path.join(INTERIM, 'coco_merged.json')
with open(merged_path,'w') as f: json.dump(merged,f)
print(f'Merged: {len(merged["images"])} images, {len(merged["annotations"])} annotations')
print(f'Categories: {[c["name"] for c in merged["categories"]]}')

In [ ]:
# ── Create splits (by identity or by image) ──
from collections import defaultdict

def get_identity_map(coco):
    id2imgs = defaultdict(set)
    for ann in coco['annotations']:
        attrs = ann.get('attributes',{})
        ident = attrs.get('identity_id', f"auto_{ann['image_id']}")
        if ident=='NA': ident = f"auto_{ann['image_id']}"
        id2imgs[ident].add(ann['image_id'])
    return dict(id2imgs)

SEED = 42
TRAIN_R, VAL_R, TEST_R = 0.70, 0.15, 0.15

id_map = get_identity_map(merged)
identities = sorted(id_map.keys())
random.seed(SEED); random.shuffle(identities)

n = len(identities)
nt = int(n*TRAIN_R); nv = int(n*VAL_R)

splits = {'train':[],'val':[],'test':[]}
for i in identities[:nt]:      splits['train'].extend(id_map[i])
for i in identities[nt:nt+nv]: splits['val'].extend(id_map[i])
for i in identities[nt+nv:]:   splits['test'].extend(id_map[i])
for k in splits: splits[k] = sorted(set(splits[k]))

splits_path = os.path.join(MANIFESTS, 'dataset_splits.json')
with open(splits_path,'w') as f: json.dump(splits,f,indent=2)

print(f'Splits saved to {splits_path}')
print(f'  train: {len(splits["train"])} images')
print(f'  val:   {len(splits["val"])} images')
print(f'  test:  {len(splits["test"])} images')
print(f'  identities: {len(identities)}')

In [ ]:
# ── Dataset stats ──
bbox_areas = [a['area'] for a in merged['annotations']]
bbox_w = [a['bbox'][2] for a in merged['annotations']]
bbox_h = [a['bbox'][3] for a in merged['annotations']]

stats = pd.DataFrame([{
    'total_images': len(merged['images']),
    'total_annotations': len(merged['annotations']),
    'categories': len(merged['categories']),
    'identities': len(identities),
    'train_images': len(splits['train']),
    'val_images': len(splits['val']),
    'test_images': len(splits['test']),
    'avg_bbox_area': np.mean(bbox_areas) if bbox_areas else 0,
    'avg_bbox_w': np.mean(bbox_w) if bbox_w else 0,
    'avg_bbox_h': np.mean(bbox_h) if bbox_h else 0,
    'min_bbox_area': np.min(bbox_areas) if bbox_areas else 0,
    'max_bbox_area': np.max(bbox_areas) if bbox_areas else 0,
    'avg_ann_per_img': len(merged['annotations'])/max(len(merged['images']),1),
}]).T.rename(columns={0:'value'})

stats_path = os.path.join(MANIFESTS, 'dataset_stats.csv')
stats.to_csv(stats_path)
print(stats)

---
## 3. Convert to YOLO format

In [ ]:
YOLO_DIR = os.path.join(PROCESSED, 'yolo_dataset')
SINGLE_CLASS = True
CLASS_NAME = 'person'   # change to 'license_plate' if needed

# Create dirs
for sp in ['train','val','test']:
    os.makedirs(os.path.join(YOLO_DIR,'images',sp), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR,'labels',sp), exist_ok=True)

img_lookup = {img['id']:img for img in merged['images']}
ann_by_img = defaultdict(list)
for ann in merged['annotations']:
    ann_by_img[ann['image_id']].append(ann)

for split_name, image_ids in splits.items():
    for iid in tqdm(image_ids, desc=f'YOLO {split_name}'):
        if iid not in img_lookup: continue
        info = img_lookup[iid]
        src = info['file_name']; w,h = info['width'],info['height']
        ext = os.path.splitext(src)[1]
        dst_img = os.path.join(YOLO_DIR,'images',split_name,f'{iid:08d}{ext}')
        if not os.path.exists(dst_img) and os.path.isfile(src):
            try: os.symlink(src, dst_img)
            except: shutil.copy2(src, dst_img)
        
        dst_lbl = os.path.join(YOLO_DIR,'labels',split_name,f'{iid:08d}.txt')
        with open(dst_lbl,'w') as f:
            for ann in ann_by_img.get(iid,[]):
                bx,by,bw,bh = ann['bbox']
                cx = max(0,min(1,(bx+bw/2)/w))
                cy = max(0,min(1,(by+bh/2)/h))
                nw = max(0,min(1,bw/w))
                nh = max(0,min(1,bh/h))
                ci = 0 if SINGLE_CLASS else ann['category_id']-1
                f.write(f'{ci} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')

# Write dataset.yaml
names = [CLASS_NAME] if SINGLE_CLASS else [c['name'] for c in merged['categories']]
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val: images/val
test: images/test

nc: {len(names)}
names: {names}
"""
DATASET_YAML = os.path.join(YOLO_DIR, 'dataset.yaml')
with open(DATASET_YAML,'w') as f: f.write(yaml_content)

print(f'\n✅ YOLO dataset ready: {DATASET_YAML}')
# Quick check
for sp in ['train','val','test']:
    ni = len(os.listdir(os.path.join(YOLO_DIR,'images',sp)))
    nl = len(os.listdir(os.path.join(YOLO_DIR,'labels',sp)))
    print(f'  {sp}: {ni} images, {nl} labels')

---
## 4. Phase 1 — Train YOLOv8n detector

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8n
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data=DATASET_YAML,
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='AdamW',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=10,
    seed=42,
    workers=4,
    # Augmentation
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    erasing=0.4,
    # Output
    project=ARTIFACTS,
    name='train_run',
    exist_ok=True,
)
print('\n✅ Training complete!')

In [ ]:
# ── Load best weights & evaluate ──
best_pt = os.path.join(ARTIFACTS, 'train_run', 'weights', 'best.pt')
if not os.path.isfile(best_pt):
    # Try fallback path
    best_pt = glob.glob(os.path.join(ARTIFACTS, '**', 'best.pt'), recursive=True)
    best_pt = best_pt[0] if best_pt else None

print(f'Best weights: {best_pt}')
model = YOLO(best_pt)

# Evaluate
val_results = model.val(
    data=DATASET_YAML,
    imgsz=640,
    conf=0.01,
    iou=0.7,
    device=0,
)

print(f'\nmAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
# Access precision/recall if available
try:
    print(f'Precision:    {val_results.box.mp:.4f}')
    print(f'Recall:       {val_results.box.mr:.4f}')
except: pass

In [ ]:
# ── Benchmark latency ──
dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
for _ in range(5): model.predict(dummy, verbose=False)

latencies = []
for _ in range(50):
    t0 = time.perf_counter()
    model.predict(dummy, verbose=False)
    latencies.append((time.perf_counter()-t0)*1000)

latencies.sort()
bench = {
    'mean_ms': np.mean(latencies),
    'p50_ms': latencies[len(latencies)//2],
    'p95_ms': latencies[int(len(latencies)*0.95)],
    'fps': 1000/np.mean(latencies),
    'model_size_mb': os.path.getsize(best_pt)/(1024*1024),
}

print('\nLatency Benchmark:')
for k,v in bench.items(): print(f'  {k}: {v:.2f}')

with open(os.path.join(ARTIFACTS,'benchmark.json'),'w') as f:
    json.dump(bench,f,indent=2)

In [ ]:
# ── Export ONNX ──
onnx_path = model.export(format='onnx', imgsz=640, simplify=True, dynamic=False, half=False)
print(f'\n✅ ONNX exported: {onnx_path}')
print(f'   Size: {os.path.getsize(onnx_path)/(1024*1024):.1f} MB')

---
## 5. Evaluation report & visualizations

In [ ]:
# ── Generate evaluation report ──
TARGET_MAP50 = 0.80
TARGET_RECALL = 0.75
TARGET_FPS = 25
TARGET_SIZE = 10

map50_val = val_results.box.map50
map_val = val_results.box.map
try: prec_val = val_results.box.mp
except: prec_val = 'N/A'
try: rec_val = val_results.box.mr
except: rec_val = 'N/A'

report = f"""# Detector Evaluation Report

## Model
- **Architecture**: YOLOv8n (single-class: {CLASS_NAME})
- **Input size**: 640
- **Confidence threshold**: 0.01 (privacy-first)

## Detection Metrics
| Metric | Value | Target | Status |
|--------|-------|--------|--------|
| mAP@0.5 | {map50_val:.4f} | ≥ {TARGET_MAP50} | {'✅' if map50_val>=TARGET_MAP50 else '❌'} |
| mAP@0.5:0.95 | {map_val:.4f} | — | — |
| Precision | {prec_val} | — | — |
| Recall | {rec_val} | ≥ {TARGET_RECALL} | {'✅' if isinstance(rec_val,float) and rec_val>=TARGET_RECALL else '❌' if isinstance(rec_val,float) else '—'} |

## Latency Benchmark
| Metric | Value | Target |
|--------|-------|--------|
| Mean latency | {bench['mean_ms']:.1f} ms | — |
| P50 latency | {bench['p50_ms']:.1f} ms | — |
| P95 latency | {bench['p95_ms']:.1f} ms | — |
| FPS | {bench['fps']:.1f} | ≥ {TARGET_FPS} |
| Model size | {bench['model_size_mb']:.1f} MB | ≤ {TARGET_SIZE} MB |

## Dataset
| Split | Images |
|-------|--------|
| Train | {len(splits['train'])} |
| Val   | {len(splits['val'])} |
| Test  | {len(splits['test'])} |

## Notes
- Conf threshold 0.01 used for eval to prioritize recall.
- False negatives (missed persons) are critical in privacy applications.
"""

report_path = os.path.join(REPORTS, 'detector_eval.md')
with open(report_path,'w') as f: f.write(report)
print(f'✅ Report saved to {report_path}')
print(report)

In [ ]:
# ── Visualize predictions on val images ──
val_img_dir = os.path.join(YOLO_DIR, 'images', 'val')
val_images = [os.path.join(val_img_dir,f) for f in os.listdir(val_img_dir)
              if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))][:10]

print(f'Visualizing {len(val_images)} val images...')

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, min(5,len(val_images)), figsize=(20,8))
if len(val_images) < 2: axes = [axes]

for idx, img_path in enumerate(val_images[:10]):
    frame = cv2.imread(img_path)
    if frame is None: continue
    
    preds = model.predict(frame, conf=0.25, verbose=False)
    annotated = preds[0].plot()
    
    # Save
    out = os.path.join(VISUALS, f'pred_{os.path.basename(img_path)}')
    cv2.imwrite(out, annotated)
    
    # Display
    r,c = idx//5, idx%5
    if len(val_images)>=2:
        ax = axes[r][c] if isinstance(axes[0],np.ndarray) else axes[c]
    else:
        ax = axes[idx]
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    n_det = len(preds[0].boxes) if preds[0].boxes is not None else 0
    ax.set_title(f'{n_det} dets')
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(VISUALS, 'val_predictions_grid.png'), dpi=150)
plt.show()
print(f'✅ Visualizations saved to {VISUALS}')

---
## 6. Summary

### Phase 0 outputs
| File | Path |
|------|------|
| COCO annotations | `data/interim/coco_*.json` |
| Merged COCO | `data/interim/coco_merged.json` |
| Splits | `data/manifests/dataset_splits.json` |
| Stats | `data/manifests/dataset_stats.csv` |
| YOLO dataset | `data/processed/yolo_dataset/dataset.yaml` |

### Phase 1 outputs
| File | Path |
|------|------|
| Best weights | `artifacts/detector/train_run/weights/best.pt` |
| ONNX model | `artifacts/detector/train_run/weights/best.onnx` |
| Benchmark | `artifacts/detector/benchmark.json` |
| Eval report | `reports/detector_eval.md` |
| Visual samples | `artifacts/visuals/` |

In [ ]:
# ── Final summary ──
print('='*70)
print('PHASE 0 + PHASE 1 COMPLETE')
print('='*70)
print(f'  Merged COCO:    {merged_path}')
print(f'  Splits:         {splits_path}')
print(f'  Stats:          {stats_path}')
print(f'  YOLO dataset:   {DATASET_YAML}')
print(f'  Best weights:   {best_pt}')
print(f'  ONNX model:     {onnx_path}')
print(f'  Eval report:    {report_path}')
print(f'\n  mAP@0.5:   {map50_val:.4f}')
print(f'  FPS:       {bench["fps"]:.1f}')
print(f'  Model:     {bench["model_size_mb"]:.1f} MB')
print(f'\nNext: Phase 2 — ByteTrack integration')